<a href="https://colab.research.google.com/github/akimotolab/CMAES_Tutorial/blob/main/2_step_size_adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 步长自适应（Step-Size Adaptation）

在前面的 Sphere 函数实验中，我们已经观察到以下现象：
1. 当步长与维数的乘积相对于均值向量到最优解的距离非常小时，从对数尺度看目标函数的下降率较低。（需要注意：如果关注的不是下降率，而是目标函数值的绝对下降量，那么这种情况下的下降量可能与第 3 种情况相当，甚至更大。这里讨论的是相对下降率。）
2. 当步长与维数的乘积相对于均值向量到最优解的距离非常大时，搜索会停滞。
3. 当步长与维数的乘积和均值向量到最优解的距离处于相近量级时，从对数尺度看目标函数的下降率较高。

出现这些现象的原因可以理解如下。

1. 当 $\|m-x^*\| \gg \sigma$ 时：均值向量通过对生成候选解中排名前 $\mu$ 个解取平均来更新。候选解以 $m$ 为中心，通过加入 $\sigma\mathcal{N}(0,I)$ 生成。由于 $\|\mathcal{N}(0,I)\|$ 服从自由度为 $N$ 的 $\chi$ 分布，其期望量级约为 $\sqrt{N}$，因此一次 $m$ 更新能够移动的距离至多约为 $\sigma\sqrt{N}$。于是更新前后到最优解的距离满足近似关系
$$
\frac{\| m' - x^* \|}{\| m - x^* \|} \gtrapprox \frac{\| m - x^* \| - \alpha \sigma \sqrt{N}}{\| m - x^* \|} = 1 - \frac{\alpha \sigma \sqrt{N}}{\| m - x^* \|}
$$
其中 $\alpha>0$ 是某个比例常数。当 $\|m-x^*\| \gg \sigma$ 时，右侧第二项接近 0，因此从比例意义上看，均值向量到最优解的距离下降得很慢。但由于每次移动量本身仍与 $\sigma\sqrt{N}$ 成比例，所以算法仍会以近似恒定的绝对步幅靠近最优解。

2. 当 $\|m-x^*\| \ll \sigma$ 时：为便于讨论，考虑 $m=x^*$。此时候选解按照 $x_i\sim\sigma\mathcal{N}(0,I)$ 生成。算法从中选出前 $\mu$ 个解并求加权平均。为了理解其尺度，先假设随机选取 $\mu$ 个解，则根据正态分布的性质有
$$
\sum_{i=1}^{\mu} w_i x_i \sim \mathcal{N}\left( 0, \left(\sum_{i=1}^{\mu} w_i^2\right)\sigma^2 I \right) \sim \frac{\sigma }{\sqrt{\mu}}\mathcal{N}\left( 0, I \right).
$$
这就是更新后均值向量 $m'$ 的典型分布尺度。也就是说，即使更新前恰好有 $m=x^*$，更新后均值向量通常也会离开最优解约 $\sigma\sqrt{N}/\sqrt{\mu}$ 的距离。实际算法选择的是更接近最优解的优秀候选解，因此真实分布会比上述随机选择更靠近最优解；不过由于高维正态分布的范数集中在其期望附近，这个尺度判断仍然成立。因此，当 $\sigma$ 固定时，算法不可能无限精细地收敛到最优解。

3. 当 $\|m-x^*\| \propto N\sigma$ 时：由第 1 点可以看出，如果 $\sigma$ 与 $\|m-x^*\|$ 成比例，均值向量就能够以指数速度接近最优解。为什么更具体地说应该满足 $\|m-x^*\|\propto N\sigma$，并不能只靠直觉简单解释。对于凸二次函数，理论分析表明，要使一次均值更新后的目标函数期望下降量最大，最优步长满足 $\sigma\propto\|\nabla f\|/Tr(\nabla\nabla f)$ [1,2]。对于 Sphere 函数，
$$
\sigma \propto \frac{\|\nabla f\|}{Tr(\nabla \nabla f)} = \frac{\| m - x^*\|}{N}.
$$

#### 参考文献
1. D. Arnold. Optimal Weighted Recombination. FOGA (2005)

2. Y. Akimoto, A. Auger, N. Hansen. Quality gain analysis of the weighted recombination evolution strategy on general convex quadratic functions. TCS (2020)

## 凸二次函数下的最优步长

如上所述，文献 [1] 针对球对称函数 $h(x)=\frac{1}{2}(x-x^*)^\mathrm{T}(x-x^*)$，文献 [2] 针对一般凸二次函数 $h(x)=\frac{1}{2}(x-x^*)^\mathrm{T}A(x-x^*)$，考虑其经过单调变换 $g:\mathbb{R}\to\mathbb{R}$ 后得到目标函数 $f(x)=g(h(x))$ 的情况，并推导了相应的最优步长。

这些工作采用归一化的目标函数期望下降率作为指标：
$$
\frac{N(f(m)-\mathbb{E}[f(m')\mid m,\sigma])}{f(m)},
$$
并在维数极限 $N\to\infty$ 下推导使该指标最大的 $\sigma$。这就是前面所说的最优步长。

对于球对称函数，最优步长与当前均值向量到最优解的距离除以维数 $N$ 成比例。虽然这一结果是在 $N\to\infty$ 的极限中导出的，但在有限维问题中也通常能很好地近似最优值。

#### 补充：关于步长应与 $N$ 成反比的常见误解

一个常见误解是认为 Sphere 函数下的最优步长应该与
$$
\frac{\|m-x^*\|}{\sqrt{N}}
$$
成比例，也就是与 $\sqrt{N}$ 而不是 $N$ 成反比。这个直觉来自：当前均值到最优解的距离是 $\|m-x^*\|$，似乎候选解到均值的典型距离也应该与它相当。由于候选解按 $x\sim m+\sigma\mathcal{N}(0,I)$ 生成，并且 $\|x-m\|=\sigma\|\mathcal{N}(0,I)\|\approx\sigma\sqrt{N}$，于是令 $\sigma=\|m-x^*\|/\sqrt{N}$ 看起来很自然。但这个推理忽略了候选解排序所依赖的目标函数值分布，因此结论并不正确。

下面用一个不追求严格性的分析说明为什么应与 $N$ 而不是 $\sqrt{N}$ 成反比。利用算法的旋转不变性和平移不变性，不失一般性地令 $x^*=0$，$m=(m_1,0,\dots,0)$。第一维对应最优解方向，其余维度与该方向正交。令候选解 $x=m+\sigma z$，其中 $z\sim N(0,I)$，并记 $e_1=(1,0,\dots,0)$、$z_\bot=z-z_1e_1$，则
$$\begin{aligned}
f(x)&=f(m)+\sigma m^\mathrm{T}z+\frac{\sigma^2}{2}z^\mathrm{T}z\\
&=f(m)+\sigma m_1z_1+\frac{\sigma^2}{2}z^\mathrm{T}z.
\end{aligned}$$
两边减去 $f(m)$ 并除以 $N$，得到
$$
\frac{f(x)-f(m)}{N}=\frac{\sigma\|m-x^*\|z_1}{N}+\frac{\sigma^2}{2}\frac{z^\mathrm{T}z}{N}.
$$
由于 $z\sim N(0,I)$，第二项中的 $z^\mathrm{T}z/N$ 的期望为 1，标准差为 $1/\sqrt{N}$；而第一项中的 $z_1$ 服从标准正态分布，标准差为 1。如果 $\sigma\in O(\|m-x^*\|/N)$，那么第二项相对于第一项的随机波动要小约 $1/\sqrt{N}$，因此候选解的排序主要反映其是否朝向最优解。反之，如果 $\sigma\in\Omega(\|m-x^*\|/\sqrt{N})$，第二项的波动会与第一项相当甚至更大，排名靠前的解就可能只是碰巧具有较小的 $\|z\|$，而未必沿最优方向。

因此，步长需要与 $N$ 成反比的核心原因，可以理解为：在候选解排序时，需要让“朝向最优解的信号”强于由高维随机范数带来的噪声。

## 使用最优步长时的行为

最优步长通常只有在目标函数属于特定形式、并且能够获得梯度信息时才能计算，因此直接使用最优步长并不是一般黑盒优化中的实用算法。不过我们仍然可以观察这种理想情况下的行为，作为理解步长自适应的参照。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class OptimalES(object):
    """将步长设置为与均值向量到最优解的距离成比例的 Evolution Strategy。"""

    def __init__(self, func, init_mean, sigma_coeff, nsample):
        """构造函数

        Parameters
        ----------
        func : callable
            目标函数（最小化）
        init_mean : ndarray (1D)
            初始均值向量
        sigma_coeff : float
            步长比例系数
        nsample : int
            样本数量
        """
        self.func = func
        self.mean = init_mean.copy()
        self.sigma_coeff = sigma_coeff
        self.N = self.mean.shape[0]                     # 搜索空间维数
        self.arx = np.zeros((nsample, self.N)) * np.nan # 候选解
        self.arf = np.zeros(nsample) * np.nan           # 候选解的目标函数值
        self.sigma = self.sigma_coeff * np.linalg.norm(self.mean) / self.N

        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # 权重，总和为 1

    def sample(self):
        """生成候选解。"""
        self.arx = self.mean + self.sigma * np.random.normal(size=self.arx.shape)

    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]):
            self.arf[i] = self.func(self.arx[i])

    def update_mean(self):
        """更新均值向量。"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        self.mean += np.dot(self.weights, (self.arx[idx] - self.mean))

    def update_sigma(self):
        """更新步长。"""
        self.sigma = self.sigma_coeff * np.linalg.norm(self.mean) / self.N

In [ ]:
def sphere(x):
    """Sphere 函数，最优解为 (0,...,0)。"""
    return np.dot(x, x) / 2

In [ ]:
es = OptimalES(func=sphere,
        init_mean=np.ones(10),
        sigma_coeff=0.01,
        nsample=10)

maxiter = 5000
fbest = np.zeros(maxiter) * np.nan
fmean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
for i in range(fbest.shape[0]):
    es.sample()
    es.evaluate()
    es.update_mean()
    es.update_sigma()
    fbest[i] = es.arf.min()
    fmean[i] = sphere(es.mean)
    sigmaN[i] = es.sigma * es.N

In [ ]:
plt.semilogy(fbest, '-r', label='f(best)')
plt.semilogy(fmean, '-b', label='f(mean)')
plt.semilogy(sigmaN, '--g', label='sigma*N')
plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend()

比较收敛速度（曲线斜率）。
$$CR := -\frac{N}{t}\log_{10}\frac{\|X^{(t)}\|}{\|X^{(0)}\|}$$
当 $CR>0$ 时表示收敛，数值越大收敛越快；当 $CR<0$ 时表示发散。

In [ ]:
for N in [10, 100, 1000]:
    sigma_co = 2 ** np.arange(-6., 3., 0.2)
    init_mean = np.ones(N)
    maxiter = 1000
    nsample = 10
    cr = np.zeros(sigma_co.shape[0])
    nseed = 20

    for j in range(sigma_co.shape[0]):
        sc = sigma_co[j]
        for seed in np.arange(nseed):
            np.random.seed(seed * 100)  # 随机算法在相同设置下重复运行多次并取平均

            es = OptimalES(func=sphere,
                          init_mean=init_mean,
                          sigma_coeff=sc,
                          nsample=nsample)

            for i in range(maxiter):
                es.sample()
                es.evaluate()
                es.update_mean()
                es.update_sigma()

            cr[j] += - (np.log10(np.linalg.norm(es.mean)) - np.log10(np.linalg.norm(init_mean))) / maxiter
        cr[j] *= es.N / nseed
        print(sc, cr[j])

    # 保存数据
    np.savetxt('CR_Sphere_N' + str(init_mean.shape[0]) + '_L' + str(nsample) + '.csv', np.vstack((sigma_co, cr)).T)

In [ ]:
data = np.loadtxt('CR_Sphere_N10_L10.csv')
plt.semilogx(data[:, 0], data[:, 1], '+r', label='N=10')
data = np.loadtxt('CR_Sphere_N100_L10.csv')
plt.semilogx(data[:, 0], data[:, 1], '*b', label='N=100')
data = np.loadtxt('CR_Sphere_N1000_L10.csv')
plt.semilogx(data[:, 0], data[:, 1], 'xg', label='N=1000')
plt.grid()
plt.legend(loc='best')
plt.ylabel('Convergence Rate')
plt.xlabel('Sigma Coefficient')

## 步长自适应

一般情况下：
* 目标函数并不是凸二次函数。
* 在黑盒优化设置中无法计算梯度，因此不能依赖梯度来更新步长。

### 累积步长自适应（Cumulative Step-size Adaptation, CSA）

#### 基本思想
* 均值向量持续朝同一方向移动 → 增大步长
* 均值向量呈随机方向移动 → 保持步长基本不变
* 均值向量来回振荡 → 减小步长

这个直觉比较容易理解：如果均值向量长期朝同一方向前进，通常意味着当前步长偏小；如果不断来回，则通常意味着步长偏大。困难之处在于，如何确定“太大”和“太小”的分界。CSA 设计中最关键的一点，是把“均值向量像随机游走一样移动”作为增大或减小步长的基准状态。

#### 为什么以随机移动作为基准
CSA 用均值向量随机移动的情况来判断当前步长是否过大或过小，这基于以下观察：

1. 如果步长恰当，那么上一代均值向量的移动方向和下一代的移动方向应当倾向于正交。否则，如果两者同向，说明上一代还可以沿相同方向走得更远；如果反向，则说明上一代走得过头。

2. 独立于上面的优化直觉，如果连续两代的均值移动可以看成两个相互独立、期望为 0 的随机向量，则它们内积的期望为 0，也就是在期望意义上相互正交。

因此，随机移动恰好对应一种“既不持续同向，也不持续反向”的基准状态。

#### 具体更新方式
此前为了简化讨论，我们主要考虑 $\Sigma=\sigma^2I$。为了后续推广，这里考虑更一般的搜索分布 $\mathcal{N}(m,\Sigma)$，候选解可以写为 $x_i\sim m+\sqrt{\Sigma}\mathcal{N}(0,I)$，其中 $\sqrt{\Sigma}$ 是满足 $\Sigma=\sqrt{\Sigma}\sqrt{\Sigma}$ 的半正定对称矩阵。

CSA 按以下方式更新：
$$
\begin{aligned}
p_\sigma &\leftarrow (1 - c_\sigma) p_\sigma + \sqrt{\frac{c_\sigma (2 - c_\sigma)}{\sum_{j=1}^{\lambda} w_j^2} } \sum_{i=1}^{\lambda} w_i \sqrt{\Sigma}^{-1} (x_{i:\lambda} - m)\\
\sigma &\leftarrow \sigma \exp\left( \frac{c_\sigma}{d_\sigma} \left( \frac{\|p_\sigma\|}{\mathbb{E}[\|\mathcal{N}(0, I)\|]} - 1\right)\right).
\end{aligned}
$$
其中 $x_{i:\lambda}$ 表示 $\lambda$ 个候选解中目标函数值排名第 $i$ 的解，$m$ 是生成这些候选解时、尚未更新的均值向量。$c_\sigma$ 称为累积因子（cumulation factor），可以理解为进化路径大约累积了与 $1/c_\sigma$ 成比例数量的迭代信息；$d_\sigma$ 称为阻尼因子（damping factor），用于防止 $\sigma$ 发生过于剧烈的变化。

#### 设计原理
$p_\sigma$ 称为进化路径（evolution path），它累积均值向量的移动。步长 $\sigma$ 根据进化路径的长度更新：若 $\|p_\sigma\|$ 大于 $N$ 维标准正态分布范数的期望 $\mathbb{E}[\|\mathcal{N}(0,I)\|]$，就增大步长；反之则减小步长。

下面说明“当均值随机移动时，步长平均不变”这一设计目标是如何由公式实现的。假设目标函数值完全随机、与 $x$ 无关。此时候选解排名也是随机的，所以 $x_{i:\lambda}$ 仍然独立服从生成它们的分布 $\mathcal{N}(m,\Sigma)$，均值向量因而表现为随机移动。

首先，根据正态分布的性质，$\sqrt{\Sigma}^{-1}(x_{i:\lambda}-m)\sim\mathcal{N}(0,I)$。这一步本质上是在对白化后的候选解进行标准化。

然后考虑
$$
\sqrt{\frac{1}{\sum_{j=1}^{\lambda} w_j^2}}\sum_{i=1}^{\lambda}w_i\sqrt{\Sigma}^{-1}(x_{i:\lambda}-m).
$$
由于各标准化向量相互独立且服从 $\mathcal{N}(0,I)$，加权平均的方差会缩小，而前面的 $\sqrt{1/\sum_j w_j^2}$ 恰好把这一方差重新归一化，使整个量仍服从 $\mathcal{N}(0,I)$。如果优秀解在某个方向上存在偏置，那么这些标准化向量的均值就不再为 0，此时这项归一化也会放大这种方向性信号。

最后看 $(1-c_\sigma)$ 与 $\sqrt{c_\sigma(2-c_\sigma)}$。记初始进化路径为 $p_\sigma^{(0)}$，更新 $t$ 次后的路径为 $p_\sigma^{(t)}$。在随机目标函数下，
$$\begin{aligned}
p_\sigma^{(t)}&\sim(1-c_\sigma)p_\sigma^{(t-1)}+\sqrt{c_\sigma(2-c_\sigma)}\mathcal{N}(0,I)\\
&\sim(1-c_\sigma)^tp_\sigma^{(0)}+\mathcal{N}\left(0,(1-(1-c_\sigma)^{2t})I\right).
\end{aligned}$$
因此，当 $t$ 足够大时，进化路径近似服从 $\mathcal{N}(0,I)$。于是随机目标函数下，对数步长更新的期望为
$$\begin{aligned}
\log(\sigma)&\leftarrow\log(\sigma)+\mathbb{E}\left[\frac{c_\sigma}{d_\sigma}\left(\frac{\|p_\sigma\|}{\mathbb{E}[\|\mathcal{N}(0,I)\|]}-1\right)\right]\\
&=\log(\sigma).
\end{aligned}$$
也就是说，对数步长在随机目标函数下是无偏的。这说明 CSA 的更新规则确实实现了前面的设计思想。

补充说明：如果目标函数完全随机，那么评价结果没有提供任何可利用的信息，此时算法参数在平均意义上不应发生系统性变化。让参数更新在这种无信息情形下保持无偏，是设计自适应优化算法时非常重要的原则。

#### 实现

In [ ]:
class CSAES(object):
    """CSA Evolution Strategy"""

    def __init__(self, func, init_mean, init_sigma, nsample):
        """构造函数

        Parameters
        ----------
        func : callable
            目标函数（最小化）
        init_mean : ndarray (1D)
            初始均值向量
        init_sigma : float
            初始步长
        nsample : int
            样本数量
        """
        self.func = func
        self.mean = init_mean
        self.sigma = init_sigma
        self.N = self.mean.shape[0]                     # 搜索空间维数
        self.arx = np.zeros((nsample, self.N)) * np.nan # 候选解
        self.arf = np.zeros(nsample) * np.nan           # 候选解的目标函数值

        self.weights = np.zeros(nsample)
        self.weights[:nsample//4] = 1.0 / (nsample//4)  # 权重，总和为 1

        # For CSA
        self.ps = np.zeros(self.N)
        self.cs = 4.0 / (self.N + 4.0)
        self.ds = 1.0 + self.cs
        self.chiN = np.sqrt(self.N) * (1.0 - 1.0 / (4.0 * self.N) + 1.0 / (21.0 * self.N * self.N))
        self.mueff = 1.0 / np.sum(self.weights**2)

    def sample(self):
        """生成候选解。"""
        self.arx = self.mean + self.sigma * np.random.normal(size=self.arx.shape)

    def evaluate(self):
        """评估候选解。"""
        for i in range(self.arf.shape[0]):
            self.arf[i] = self.func(self.arx[i])

    def update_mean(self):
        """更新均值向量。"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        self.mean += np.dot(self.weights, (self.arx[idx] - self.mean))

    def update_sigma(self):
        """CSA"""
        idx = np.argsort(self.arf)  # idx[i] 是目标函数值排名第 i 的候选解索引
        # 更新进化路径（累积均值向量的移动）
        self.ps = (1 - self.cs) * self.ps + np.sqrt(self.cs * (2 - self.cs) * self.mueff) * np.dot(self.weights, (self.arx[idx] - self.mean)) / self.sigma
        # 如果进化路径长度大于随机函数下的期望值，则增大步长
        self.sigma = self.sigma * np.exp(self.cs / self.ds * (np.linalg.norm(self.ps) / self.chiN - 1))

In [ ]:
es = CSAES(func=sphere,
           init_mean=np.ones(10),
           init_sigma=1.,
           nsample=10)

maxiter = 200
mean = np.zeros(maxiter) * np.nan
sigmaN = np.zeros(maxiter) * np.nan
for i in range(maxiter):
    es.sample()
    es.evaluate()
    es.update_sigma()
    es.update_mean()
    mean[i] = sphere(es.mean)
    sigmaN[i] = es.sigma * es.N

In [ ]:
plt.semilogy(mean, '-b', label='f(mean)')
plt.semilogy(sigmaN, '--g', label='sigma*N')
plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend()

#### 与最优步长比较

In [ ]:
# CSA
plt.semilogy(sigmaN / mean, '-b', label='CSA')
# Optimal
data = np.loadtxt('CR_Sphere_N10_L10.csv')
idx = np.argmax(data[:, 1])
plt.semilogy(np.ones(mean.shape) * data[idx, 0], '-r', label='Opt')

plt.title('sigma * N / ||M|| 10D-Sphere')
plt.xlabel('no. of iterations', fontsize='large')
plt.grid()
plt.legend(loc='best')

#### 观察结果
* 当初始步长过小时，算法需要较长时间才能学习到稳定的步长。
* CSA 学到的步长往往比理论最优步长略大。

## 思考
在解释 CSA 时，我们考虑了目标函数值完全随机的情况，此时 $x_{i:\lambda}$ 仍服从生成候选解时使用的正态分布 $\mathcal{N}(m,\Sigma)$。当目标函数不是随机函数时，经过排名和选择后的 $x_{i:\lambda}$ 的分布当然会与原始生成分布不同。假设 $x_{1:\lambda},\dots,x_{\lfloor\lambda/4\rfloor:\lambda}$ 独立服从某个 $\mathcal{N}(m^*,\Sigma^*)$，可以进一步分析进化路径会服从什么分布，以及步长更新将表现出怎样的行为。

作者曾在学生时期沿着类似思路分析 CSA 的局限，并提出改进方法：Akimoto et al., Functionally Specialized CMA-ES: A Modification of CMA-ES based on the Specialization of the Functions of Covariance Matrix Adaptation and Step Size Adaptation, GECCO 2008。